# Video Transcripts Preprocessing

In [1]:
import pandas as pd 
import numpy as np
import glob
import os
import re
import emoji
import pathlib
import nltk
from nltk.tokenize import sent_tokenize

In [ ]:
# --- papermill parameters (injected by runner) ---
transcript_name = "coach_tab"  # default; overridden by papermill

### VTT Parsing

The WebVTT (VTT) format structures transcripts into timestamped segments.
Each segment consists of a start time, end time, and one or more lines of
spoken text. Some segments span multiple lines without blank separators
(e.g., when a sentence wraps), which the parser handles by joining
consecutive text lines until the next timestamp block.

The parser extracts each segment's start time, end time, and text into
a flat DataFrame for downstream preprocessing.

In [2]:
def parse_vtt(filepath, source_name):
    with open(filepath, 'r') as f:
        content = f.read()

    # Skip WEBVTT header, match timestamp blocks with multiline text
    pattern = (
        r'(\d{2}:\d{2}:\d{2}\.\d{3}) --> (\d{2}:\d{2}:\d{2}\.\d{3})\n'
        r'(.+?)(?=\n\d{2}:\d{2}:\d{2}\.\d{3} -->|\Z)'
    )
    matches = re.findall(pattern, content, re.DOTALL)

    rows = []
    for start, end, text in matches:
        text = ' '.join(text.strip().split('\n')).strip()
        if text:
            rows.append({
                'source': source_name,
                'start_time': start,
                'end_time': end,
                'raw_text': text,
            })

    return pd.DataFrame(rows)

In [ ]:
# Use the injected transcript_name parameter

# Build path dynamically
vtt_path = f'../../data/video-transcripts/transcripts-{transcript_name}/timestamped.vtt'
transcript_df = parse_vtt(vtt_path, transcript_name)

print(f"Parsed {len(transcript_df)} segments from '{transcript_name}'")
transcript_df.head()

Parsed 362 segments from 'ateneo_presscon'


,source,start_time,end_time,raw_text
0,ateneo_presscon,00:00:00.000,00:00:06.000,Good afternoon and thank you for being with us...
1,ateneo_presscon,00:00:06.000,00:00:16.000,I am Matak Villanueva on behalf of the univers...
2,ateneo_presscon,00:00:16.000,00:00:25.000,to grieve the loss of our student athletes Ren...
3,ateneo_presscon,00:00:25.000,00:00:32.000,We have gathered today to address this tragedy...
4,ateneo_presscon,00:00:32.000,00:00:38.000,and discuss the measures now being undertaken ...


### Text Cleaning Pipeline

A dedicated text cleaning function was defined to standardize transcript
text prior to analysis. The following operations were applied in sequence:

- **Lowercasing**: All text was converted to lowercase to ensure
  case-insensitive matching.
- **Emoji Demojization**: Emojis were converted to their textual descriptions
  (e.g., ":crying_face:") using the `emoji` library, preserving emotional
  signals present in the text.
- **URL Removal**: Any HTTP(S) or www URLs were stripped to reduce noise.
- **Character Repetition Normalization**: Repeated characters (e.g.,
  "sooooo") were reduced to two occurrences to normalize informal emphasis
  while retaining readability.
- **Whitespace Cleanup**: Extra whitespace was collapsed and leading/trailing
  spaces were trimmed.

In [ ]:
def clean_text_for_crisis(text):
    if not isinstance(text, str):
        return ""

    # 1. Lowercase
    text = text.lower()

    # 2. Convert emojis to text tokens
    text = emoji.demojize(text)

    # 3. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 4. Normalize character repetition
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 5. Final whitespace cleanup
    text = re.sub(r'\s+', ' ', text).strip()

    return text


transcript_df['clean_text'] = transcript_df['raw_text'].apply(clean_text_for_crisis)
transcript_df[['raw_text', 'clean_text']].head()

In [ ]:
transcript_df['is_shouting'] = transcript_df['raw_text'].apply(
    lambda x: 1 if isinstance(x, str) and x.isupper() and len(x) > 5 else 0
)
transcript_df[['raw_text', 'clean_text', 'is_shouting']].head()

### Preprocessing Validation

A sample of the preprocessed segments was inspected to verify the
correctness of the cleaning pipeline. The original and cleaned text
were compared side-by-side, with particular attention to:

- **Shouting detection**: Verifying that ALL-CAPS segments were flagged
  correctly.
- **General quality**: Ensuring the cleaned text retained readability
  and meaning.

In [ ]:
sample = transcript_df.sample(10, random_state=42)[['raw_text', 'clean_text', 'is_shouting']]
sample

### Sentence Segmentation

Long segments that may exceed the 512-token limit of transformer models
are split into individual sentences via NLTK's `sent_tokenize`. Each
segment retains a `sentences` list for downstream per-sentence scoring.

In [ ]:
nltk.download('punkt_tab', quiet=True)

transcript_df['sentences'] = transcript_df['clean_text'].apply(sent_tokenize)

long_mask = transcript_df['clean_text'].str.len() > 500
print(f"Segments exceeding ~500 chars: {long_mask.sum()} ({long_mask.mean():.1%})")
transcript_df.head()

### Dataset Summary and Export

The preprocessed dataset was summarized and exported to the `output/`
directory as CSV for downstream consumption.

In [ ]:
def time_to_seconds(t):
    h, m, s = t.split(':')
    return int(h) * 3600 + int(m) * 60 + float(s)

transcript_df['duration_sec'] = (
    transcript_df['end_time'].apply(time_to_seconds) -
    transcript_df['start_time'].apply(time_to_seconds)
)

print("--- Preprocessed Dataset Summary ---")
print(f"Total segments: {len(transcript_df)}")
total_min = transcript_df['duration_sec'].sum() / 60
print(f"Total speaking duration: {total_min:.1f} min")
print(f"Shouting segments (%): {(transcript_df['is_shouting'].mean() * 100):.1f}%")

text_lens = transcript_df['clean_text'].str.len()
print(f"\n--- Text Length Stats ---")
print(f"Mean: {text_lens.mean():.0f} chars, median: {text_lens.median():.0f}")

vocab = set(' '.join(transcript_df['clean_text'].dropna()).split())
print(f"Vocabulary size: {len(vocab)}")

nulls = transcript_df.isnull().sum()
if nulls.any():
    print(f"\n--- Null Values ---")
    print(nulls[nulls > 0])
else:
    print("\nNo null values in any column")

output_dir = pathlib.Path('../output/transcripts')
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / f'transcripts-{transcript_name}.csv'

transcript_df = transcript_df.drop(columns=['raw_text', 'duration_sec'])
transcript_df = transcript_df.rename(columns={'clean_text': 'text'})
transcript_df = transcript_df[['source', 'start_time', 'end_time', 'text', 'is_shouting', 'sentences']]
transcript_df.to_csv(csv_path, index=False)

print(f"\nExported: {csv_path.resolve()}")